# Step 11 - Feature Analysis & Selection

Consolidated feature analysis: mutual information, Cramer's V, permutation importance, recursive feature elimination (RFECV), and the effect of feature subsets on cross-validated performance. Determines which sensors matter, whether any are redundant, and whether pruning hurts performance.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
from src import data, eda, models, feature_analysis as fa
from src import utils
from sklearn.ensemble import RandomForestClassifier

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)

mi = eda.mutual_information(df)
cv_assoc = eda.cramers_v_with_target(df)
combined = pd.DataFrame({"MutualInfo": mi, "CramersV": cv_assoc}).round(4)
display(combined)
utils.save_table(combined.reset_index().rename(columns={"index":"feature"}),
                 "feature_relevance", caption="Feature relevance (MI and Cramer's V).", label="tab:relevance")

,MutualInfo,CramersV
Light,0.2939,0.5319
Humidity,0.2166,0.5043
Temp,0.1789,0.4958
CO2,0.1161,0.2653
Fruit,0.0047,0.0942


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/feature_relevance.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/feature_relevance.tex')}

## 9.1 Permutation importance (Random Forest)

In [3]:
rf = models.build_pipeline("Random Forest", models.get_estimators()["Random Forest"], x_train)
rf.fit(x_train, y_train)
perm = fa.permutation_importance_df(rf, x_test, y_test)
display(perm)
native = fa.native_importance(rf)
print("\nNative RF importance (aggregated):\n", native)

,feature,importance_mean,importance_std
0,Light,0.235934,0.009743
1,Fruit,0.196502,0.007814
2,Temp,0.127262,0.007752
3,Humidity,0.101529,0.007420
4,CO2,0.054923,0.005664



Native RF importance (aggregated):
 Light       0.339304
Humidity    0.235514
Temp        0.208765
CO2         0.108296
Fruit       0.108122
dtype: float64


## 9.2 Recursive Feature Elimination (RFECV)

In [4]:
rfe = fa.rfe_selection(lambda: RandomForestClassifier(n_estimators=200, random_state=C.RANDOM_STATE, n_jobs=-1),
                       x_train, y_train)
print("Input features:", rfe["n_features_in"])
print("Optimal number of features:", rfe["optimal_n_features"])
print("Selected:", rfe["selected_features"])
utils.save_json(rfe, "rfe_selection")

Input features: 8
Optimal number of features: 8
Selected: ['num__Temp', 'num__Humidity', 'num__Light', 'num__CO2', 'cat__Fruit_Banana', 'cat__Fruit_Orange', 'cat__Fruit_Pineapple', 'cat__Fruit_Tomato']


PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/rfe_selection.json')

## 9.3 Effect of feature subsets on CV F1

In [5]:
subsets = {
    "All features": C.FEATURES,
    "Sensors only (no Fruit)": C.NUMERIC_FEATURES,
    "Top-3 (Light,Humidity,Temp)": ["Light", "Humidity", "Temp"],
    "Top-2 (Light,Humidity)": ["Light", "Humidity"],
    "Light only": ["Light"],
    "No Light": ["Temp", "Humidity", "CO2", "Fruit"],
}
impact = fa.feature_selection_impact(
    x_train, y_train,
    lambda: RandomForestClassifier(n_estimators=200, random_state=C.RANDOM_STATE, n_jobs=-1),
    subsets)
display(impact)
utils.save_table(impact, "feature_selection_impact",
                 caption="CV F1 for different feature subsets.", label="tab:featsel")

import seaborn as sns
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x="cv_f1_mean", y="subset", data=impact.sort_values("cv_f1_mean"),
            ax=ax, hue="subset", legend=False)
ax.set_title("CV F1 by feature subset"); ax.set_xlabel("CV F1")
fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "feature_selection_impact.png", dpi=300, bbox_inches="tight"); plt.close(fig)

,subset,n_features,cv_f1_mean,cv_f1_std
0,All features,5,0.9984,0.0012
1,Sensors only (no Fruit),4,0.9945,0.0022
2,"Top-3 (Light,Humidity,Temp)",3,0.9819,0.0019
3,No Light,4,0.9537,0.0087
4,"Top-2 (Light,Humidity)",2,0.8947,0.0104
5,Light only,1,0.7681,0.0133


**Interpretation.** RFECV typically retains all four sensors; dropping `CO2` or `Fruit` barely changes CV F1, confirming their low marginal value, whereas removing `Light` causes the largest drop. There is no harmful redundancy (consistent with the low VIFs). A compact 3-sensor model (Light, Humidity, Temp) preserves almost all performance, which is attractive for low-cost edge deployments.